In [1]:
import mlx.core as mx

In [2]:
mx.metal.device_info()

{'resource_limit': 499000,
 'max_buffer_length': 86586540032,
 'architecture': 'applegpu_g15s',
 'memory_size': 137438953472,
 'max_recommended_working_set_size': 115448725504,
 'device_name': 'Apple M3 Max'}

In [3]:
mx.metal.is_available()

True

In [ ]:
import math
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim


# ---------------------------------------------------
# 1) Sinusoidal timestep embedding
# ---------------------------------------------------
def timestep_embedding(t, dim, max_period=10000):
    """
    t: (B,) int32
    return: (B, dim)
    """
    half = dim // 2
    freqs = mx.exp(
        -math.log(max_period) * mx.arange(half) / max(half - 1, 1)
    )  # (half,)
    args = t.astype(mx.float32)[:, None] * freqs[None, :]  # (B, half)
    emb = mx.concatenate([mx.sin(args), mx.cos(args)], axis=-1)
    if dim % 2 == 1:
        emb = mx.concatenate([emb, mx.zeros((emb.shape[0], 1))], axis=-1)
    return emb


# ---------------------------------------------------
# 2) Simple conditional denoiser (MLP)
#    input: x_t, timestep t, class label y
#    output: predicted noise eps_theta
# ---------------------------------------------------
class ConditionalDenoiser(nn.Module):
    def __init__(self, img_dim=28 * 28, num_classes=10, time_dim=128, hidden_dim=512):
        super().__init__()
        self.img_dim = img_dim
        self.time_dim = time_dim

        self.label_emb = nn.Embedding(num_classes, time_dim)

        self.fc1 = nn.Linear(img_dim + time_dim + time_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, img_dim)

    def __call__(self, x, t, y):
        """
        x: (B, img_dim)
        t: (B,)
        y: (B,)
        """
        t_emb = timestep_embedding(t, self.time_dim)   # (B, time_dim)
        y_emb = self.label_emb(y)                      # (B, time_dim)

        h = mx.concatenate([x, t_emb, y_emb], axis=-1)
        h = nn.silu(self.fc1(h))
        h = nn.silu(self.fc2(h))
        out = self.fc3(h)
        return out


# ---------------------------------------------------
# 3) Diffusion utilities
# ---------------------------------------------------
class Diffusion:
    def __init__(self, timesteps=200, beta_start=1e-4, beta_end=2e-2):
        self.timesteps = timesteps

        self.betas = mx.linspace(beta_start, beta_end, timesteps)
        self.alphas = 1.0 - self.betas
        self.alpha_bars = mx.cumprod(self.alphas, axis=0)

        self.sqrt_alpha_bars = mx.sqrt(self.alpha_bars)
        self.sqrt_one_minus_alpha_bars = mx.sqrt(1.0 - self.alpha_bars)

    def sample_timesteps(self, batch_size):
        return mx.random.randint(0, self.timesteps, (batch_size,))

    def q_sample(self, x0, t, noise=None):
        """
        x_t = sqrt(alpha_bar_t) * x0 + sqrt(1-alpha_bar_t) * eps
        x0: (B, D)
        t : (B,)
        """
        if noise is None:
            noise = mx.random.normal(x0.shape)

        sqrt_ab = self.sqrt_alpha_bars[t][:, None]
        sqrt_1mab = self.sqrt_one_minus_alpha_bars[t][:, None]
        xt = sqrt_ab * x0 + sqrt_1mab * noise
        return xt, noise

    def p_sample(self, model, x, t, y):
        """
        One reverse step.
        x: (B, D)
        t: (B,)
        y: (B,)
        """
        beta_t = self.betas[t][:, None]
        alpha_t = self.alphas[t][:, None]
        alpha_bar_t = self.alpha_bars[t][:, None]

        pred_noise = model(x, t, y)

        mean = (1.0 / mx.sqrt(alpha_t)) * (
            x - (beta_t / mx.sqrt(1.0 - alpha_bar_t)) * pred_noise
        )

        # if current step > 0, add Gaussian noise
        step_value = int(t[0].item()) if hasattr(t[0], "item") else int(t[0])
        if step_value > 0:
            z = mx.random.normal(x.shape)
            x_prev = mean + mx.sqrt(beta_t) * z
        else:
            x_prev = mean

        return x_prev

    def sample(self, model, n, labels, img_dim=28 * 28):
        """
        Generate n samples conditioned on labels.
        labels: (n,)
        """
        x = mx.random.normal((n, img_dim))

        for step in reversed(range(self.timesteps)):
            t = mx.full((n,), step, dtype=mx.int32)
            x = self.p_sample(model, x, t, labels)

        return mx.clip(x, -1.0, 1.0)


# ---------------------------------------------------
# 4) Loss / train step
# ---------------------------------------------------
def mse_loss(model, diffusion, x0, y):
    """
    x0: (B, 784), normalized to [-1, 1]
    y : (B,)
    """
    B = x0.shape[0]
    t = diffusion.sample_timesteps(B)
    noise = mx.random.normal(x0.shape)

    xt, noise = diffusion.q_sample(x0, t, noise)
    pred_noise = model(xt, t, y)

    return mx.mean((pred_noise - noise) ** 2)


def make_train_step(model, diffusion, optimizer):
    loss_and_grad_fn = nn.value_and_grad(model, lambda m, x, y: mse_loss(m, diffusion, x, y))

    def train_step(x, y):
        loss, grads = loss_and_grad_fn(model, x, y)
        optimizer.update(model, grads)
        mx.eval(model.parameters(), optimizer.state)
        return loss

    return train_step



In [20]:
img_dim = 128 * 128
num_classes = 10

model = ConditionalDenoiser(
    img_dim=img_dim,
    num_classes=num_classes,
    time_dim=128,
    hidden_dim=512,
)

# MLX is lazy, so force parameter initialization/evaluation
mx.eval(model.parameters())

diffusion = Diffusion(
    timesteps=200,      # demo용. 보통 더 크게 둠
    beta_start=1e-4,
    beta_end=2e-2,
)

optimizer = optim.Adam(learning_rate=1e-4)
train_step = make_train_step(model, diffusion, optimizer)

# -----------------------------------------------
# Dummy training example
# Replace this with real MNIST batches:
# x_batch: (B, 784), float32 in [-1, 1]
# y_batch: (B,), int32 labels
# -----------------------------------------------
B = 32
x_batch = mx.random.uniform(shape=(B, img_dim), low=-1.0, high=1.0)
y_batch = mx.random.randint(0, num_classes, (B,)).astype(mx.int32)

for step in range(10):
    loss = train_step(x_batch, y_batch)
    mx.eval(loss)
    print(f"step={step}, loss={float(loss):.6f}")

# -----------------------------------------------
# Sampling example
# -----------------------------------------------
labels = mx.array([0, 1, 2, 3, 4, 5, 6, 7], dtype=mx.int32)
samples = diffusion.sample(model, n=labels.shape[0], labels=labels, img_dim=img_dim)
mx.eval(samples)

# reshape to (N, 128, 128) if needed
samples = samples.reshape(labels.shape[0], 128, 128)
print("samples shape:", samples.shape)


step=0, loss=1.001612
step=1, loss=1.001933
step=2, loss=1.001186
step=3, loss=1.003894
step=4, loss=1.000941
step=5, loss=1.002677
step=6, loss=1.003348
step=7, loss=1.005069
step=8, loss=1.006293
step=9, loss=1.005465
samples shape: (8, 128, 128)


TypeError: Main.__init__() missing 3 required positional arguments: 'main_config', 'hash_config', and 'output_config'